Finding 1 — Late delivery vs on-time delivery

H₀: P(1-star | late) = P(1-star | on-time)
    The proportion of 1-star reviews is the same among late-delivered 
    and on-time-delivered orders.

H₁: P(1-star | late) ≠ P(1-star | on-time)
    The proportion of 1-star reviews differs between late-delivered 
    and on-time-delivered orders.

Non-directional. Although exploration suggests late deliveries have 
higher 1-star rates, using exploration findings to choose direction 
constitutes p-hacking. We test whether the rates differ and report 
the direction separately.

Finding 2 - Multi-Seller Orders vs single-seller orders

H0: P(1-star | multi-seller) = P(1-star | single-seller)
The proportion of 1-star reviews is the same among multi-seller and single-seller orders

H1: P(1-star | multi-seller) ≠ P(1-star | single-seller)
The proportion of 1-star reviews differs between multi-seller and single-seller orders

Non-directional. Although exploration suggests multi-seller have 
higher 1-star rates, using exploration findings to choose direction 
constitutes p-hacking. We test whether the rates differ and report 
the direction separately.


Finding 3 — Product Category vs 1-Star Rate

H₀: P(1-star) is independent of product category.
    The 1-star review rate is the same across all categories.

H₁: P(1-star) is not independent of product category.
    The 1-star review rate varies by category.

Non-directional. The test asks whether category is associated with 
1-star outcomes, not which categories are higher or lower. Direction 
and specific category identification are post-hoc analyses, conducted 
only if the omnibus test is significant.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

analysis_df = pd.read_csv('../data/analysis_df.csv')

# Reconstruct was_late (CSV doesn't preserve derived columns reliably)
analysis_df['order_purchase_timestamp'] = pd.to_datetime(analysis_df['order_purchase_timestamp'])
analysis_df['order_delivered_customer_date'] = pd.to_datetime(analysis_df['order_delivered_customer_date'])
analysis_df['order_estimated_delivery_date'] = pd.to_datetime(analysis_df['order_estimated_delivery_date'])
analysis_df['delivery_delay_days'] = (
    analysis_df['order_delivered_customer_date'] - analysis_df['order_estimated_delivery_date']
).dt.days
analysis_df['was_late'] = analysis_df['delivery_delay_days'] > 0
analysis_df['is_1_star'] = analysis_df['review_score'] == 1

In [2]:
# --- Finding 1: Late Delivery ---

# Contigency table
contigency_1 = pd.crosstab(
    analysis_df['was_late'], 
    analysis_df['is_1_star'],
    margins=False
)

print("Contigency table (was_late * is_1_star):")
print(contigency_1)

#Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contigency_1)

# Odds ratio: (a*d) / (b*c) where the 2*2 table is [[a,b], [c,d]]
# a = not_late & not_1star, b = not_late & 1star
# c = late & not_1star, d =  late & 1star

a = contigency_1.iloc[0, 0]
b = contigency_1.iloc[0, 1]
c = contigency_1.iloc[1, 0]
d = contigency_1.iloc[1, 1]

odds_ratio = (a*d) / (b*c)

# 1-star rate by group (for interpretation)
late_rate = d / (c + d)
ontime_rate = b / (a + b)

print(f"\nChi-square: {chi2:.2f}")
print(f"p-value: {p_value:.2e}")
print(f"Odds ratio: {odds_ratio:.2f}")
print(f"1-star rate (late): {late_rate*100:.1f}%")
print(f"1-star rate (on-time): {ontime_rate*100:.1f}%")

Contigency table (was_late * is_1_star):
is_1_star  False  True 
was_late               
False      83482   5947
True        2947   3433

Chi-square: 14990.43
p-value: 0.00e+00
Odds ratio: 16.35
1-star rate (late): 53.8%
1-star rate (on-time): 6.6%


Late delivery is overwhelmingly associated with 1-star reviews. Late-delivered orders are 16x more likely to receive 1-star than on-time orders (53.8% vs 6.6%). The result is statistically significant beyond conventional thresholds (p ≈ 0).

## Finding 1 — Late Delivery (Test Results)

**Chi-square test of independence:**
- Test statistic: χ² = 14,990 (df = 1)
- p < 0.001
- **Odds ratio: 16.35** — late-delivered orders have 16x the odds of receiving 1-star

**Effect:**
- 1-star rate among late orders: 53.8%
- 1-star rate among on-time orders: 6.6%
- Absolute difference: 47.2 percentage points

**Conclusion:** We reject H₀. The proportion of 1-star reviews differs 
dramatically between late and on-time deliveries — this is the strongest 
effect identified in the analysis.

**Caveat:** This is observational data. The association may reflect 
confounding with order complexity, category, or seller — explored 
through stratified analysis below.

In [3]:
# Single binary variable for "is this a multi-seller order?"
analysis_df['multi_seller'] = analysis_df['unique_sellers'] > 1

# Cross-tab multi_seller vs is_1_star (the outcome)
contingency_2 = pd.crosstab(
    analysis_df['multi_seller'],
    analysis_df['is_1_star']
)
print("Contingency table (multi_seller × is_1_star):")
print(contingency_2)

chi2, p_value, dof, expected = chi2_contingency(contingency_2)

# 2x2 table layout:
# [[ single & not_1star, single & 1star ],
#  [ multi  & not_1star, multi  & 1star ]]
a = contingency_2.iloc[0, 0]  # single, not 1-star
b = contingency_2.iloc[0, 1]  # single, 1-star
c = contingency_2.iloc[1, 0]  # multi, not 1-star
d = contingency_2.iloc[1, 1]  # multi, 1-star

odds_ratio = (a * d) / (b * c)
multi_rate = d / (c + d)
single_rate = b / (a + b)

print(f"\nChi-square: {chi2:.2f}")
print(f"p-value: {p_value:.2e}")
print(f"Odds ratio: {odds_ratio:.2f}")
print(f"1-star rate (multi-seller): {multi_rate*100:.1f}%")
print(f"1-star rate (single-seller): {single_rate*100:.1f}%")

Contingency table (multi_seller × is_1_star):
is_1_star     False  True 
multi_seller              
False         85622   8926
True            807    454

Chi-square: 991.14
p-value: 1.52e-217
Odds ratio: 5.40
1-star rate (multi-seller): 36.0%
1-star rate (single-seller): 9.4%


## Finding 2 — Multi-Seller Orders (Test Results)

**Chi-square test of independence:**
- χ² = 991.14 (df = 1)
- p < 0.001
- **Odds ratio: 5.40** — multi-seller orders have 5.4x the odds of 1-star

**Effect:**
- 1-star rate among multi-seller orders: 36.0%
- 1-star rate among single-seller orders: 9.4%
- Multi-seller orders are 1.3% of volume but contribute disproportionately to 1-star reviews

**Conclusion:** We reject H₀. Multi-seller orders are significantly more 
likely to receive 1-star reviews, even after considering that they're a 
small fraction of total volume.

**Important context (from exploration):** This effect is NOT explained 
by delivery delay. Multi-seller orders actually arrive faster and more 
on-time than single-seller orders. This confirms multi-seller as an 
independent driver of dissatisfaction, separate from delivery experience.

In [4]:
# Filter out missing categories
df_cat = analysis_df.dropna(subset=['product_category_name_english']).copy()

# Contingency table
contingency_3 = pd.crosstab(
    df_cat['product_category_name_english'],
    df_cat['is_1_star']
)
print(f"Table shape: {contingency_3.shape}")

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_3)

# Cramér's V effect size
n = contingency_3.values.sum()
min_dim = min(contingency_3.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"\nChi-square: {chi2:.2f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.2e}")
print(f"Cramér's V: {cramers_v:.3f}")


Table shape: (71, 2)

Chi-square: 363.19
Degrees of freedom: 70
p-value: 3.66e-41
Cramér's V: 0.062


## Finding 3 — Product Category (Test Results)

**Chi-square test of independence:**
- χ²(70) = 363.19
- p < 0.001
- **Cramér's V = 0.062** — statistically significant but very weak association

**Conclusion:** We reject H₀. Product category is associated with 1-star 
outcomes, but the effect is small relative to delivery and multi-seller 
factors. The association is driven primarily by a handful of outlier 
categories (high-complexity items like office furniture, low-stakes items 
like food and books).

**Practical implication:** Category cannot be the primary intervention target. 
Category-specific interventions would address a narrow slice of dissatisfaction. 
Late delivery (OR=16.35) and multi-seller orders (OR=5.40) are higher-leverage 
intervention targets.